# Notebook 2: Data Preprocessing
### Sarcasm Detection in News Headlines — NLP Project
---
**Objective:** Prepare three versions of the data:
- `cleaned_text` → TF-IDF (Traditional ML: NB, LR, SVM)
- `raw_text` (light clean) → BiLSTM sequences
- `raw_text` (untouched) → BERT (handles its own tokenization)

**Key insight from dataset:** Headlines are only 8–15 words — MAX_LEN=40 covers 99% of all headlines perfectly.

## 1. Imports

In [15]:
import pandas as pd
import numpy as np
import re
import string
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import torch


from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Custom tokenizer using torch and sklearn
# (Not needed - using real TensorFlow Keras Tokenizer above)

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)

RANDOM_STATE = 42
print('All imports successful!')

All imports successful!


[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1006)>
[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1006)>
[nltk_data] Error loading punkt_tab: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1006)>
[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1006)>


## 2. Load Data

In [2]:
df = pd.read_csv('../data/sarcasm_clean.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nClass distribution:')
print(df['is_sarcastic'].value_counts())
df.head()

Shape: (28619, 4)
Columns: ['headline', 'is_sarcastic', 'word_count', 'char_count']

Class distribution:
is_sarcastic
0    14985
1    13634
Name: count, dtype: int64


,headline,is_sarcastic,word_count,char_count
0,thirtysomething scientists unveil doomsday clo...,1,8,61
1,dem rep. totally nails why congress is falling...,0,13,79
2,eat your veggies: 9 deliciously different recipes,0,7,49
3,inclement weather prevents liar from getting t...,1,8,52
4,mother comes pretty close to using word 'strea...,1,9,61


## 3. Text Cleaning Functions

In [5]:
# Keep negation words — critical for sarcasm detection
# e.g. "not bad" vs "bad" have opposite meanings
standard_stop = set(stopwords.words('english'))
keep_words    = {'no', 'not', 'nor', 'never', 'none', 'nobody',
                 'nothing', 'neither', 'hardly', 'barely', 'scarcely'}
custom_stop   = standard_stop - keep_words
lemmatizer    = WordNetLemmatizer()

# ── For TF-IDF (Traditional ML) ──────────────────────────────────
def clean_for_tfidf(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)       # remove URLs
    text = re.sub(r'\d+', '', text)                   # remove numbers
    text = text.translate(
        str.maketrans('', '', string.punctuation))    # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens
              if t not in custom_stop and len(t) > 1]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# ── For BiLSTM (light clean — keep word order & structure) ───────
def clean_for_bilstm(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)          # keep alphanumeric only
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Example
sample = df['headline'].iloc[0]
print(f'Original : {sample}')
print(f'TF-IDF   : {clean_for_tfidf(sample)}')
print(f'BiLSTM   : {clean_for_bilstm(sample)}')

Original : thirtysomething scientists unveil doomsday clock of hair loss
TF-IDF   : thirtysomething scientist unveil doomsday clock hair loss
BiLSTM   : thirtysomething scientists unveil doomsday clock of hair loss


In [6]:
df['cleaned_tfidf']  = df['headline'].apply(clean_for_tfidf)
df['cleaned_bilstm'] = df['headline'].apply(clean_for_bilstm)

# Drop any empty rows after cleaning (very rare)
df = df[
    (df['cleaned_tfidf'].str.strip().str.len()  > 0) &
    (df['cleaned_bilstm'].str.strip().str.len() > 0)
].reset_index(drop=True)

print(f'After cleaning — shape: {df.shape}')
df[['headline', 'cleaned_tfidf', 'cleaned_bilstm', 'is_sarcastic']].head()

After cleaning — shape: (28617, 6)


,headline,cleaned_tfidf,cleaned_bilstm,is_sarcastic
0,thirtysomething scientists unveil doomsday clo...,thirtysomething scientist unveil doomsday cloc...,thirtysomething scientists unveil doomsday clo...,1
1,dem rep. totally nails why congress is falling...,dem rep totally nail congress falling short ge...,dem rep totally nails why congress is falling ...,0
2,eat your veggies: 9 deliciously different recipes,eat veggie deliciously different recipe,eat your veggies 9 deliciously different recipes,0
3,inclement weather prevents liar from getting t...,inclement weather prevents liar getting work,inclement weather prevents liar from getting t...,1
4,mother comes pretty close to using word 'strea...,mother come pretty close using word streaming ...,mother comes pretty close to using word stream...,1


## 4. Word Count After Cleaning

In [7]:
df['wc_original'] = df['headline'].apply(lambda x: len(str(x).split()))
df['wc_tfidf']    = df['cleaned_tfidf'].apply(lambda x: len(str(x).split()))
df['wc_bilstm']   = df['cleaned_bilstm'].apply(lambda x: len(str(x).split()))

print('=== Word Count Summary ===')
print(df[['wc_original', 'wc_tfidf', 'wc_bilstm']].describe().round(2))
print(f'\n95th percentile (original): {df["wc_original"].quantile(0.95):.0f} words')
print(f'99th percentile (original): {df["wc_original"].quantile(0.99):.0f} words')
print(f'Max (original)            : {df["wc_original"].max()} words')

=== Word Count Summary ===
       wc_original  wc_tfidf  wc_bilstm
count     28617.00  28617.00   28617.00
mean         10.05      7.18      10.04
std           3.39      2.44       3.38
min           2.00      1.00       2.00
25%           8.00      6.00       8.00
50%          10.00      7.00      10.00
75%          12.00      9.00      12.00
max         151.00    106.00     151.00

95th percentile (original): 16 words
99th percentile (original): 19 words
Max (original)            : 151 words


## 5. Train / Val / Test Split — 70 / 15 / 15

In [8]:
X_tfidf  = df['cleaned_tfidf']
X_bilstm = df['cleaned_bilstm']
X_raw    = df['headline']           # raw text for BERT
y        = df['is_sarcastic']

# First split: 70% train, 30% temp
(
    X_tfidf_train,  X_tfidf_temp,
    X_bilstm_train, X_bilstm_temp,
    X_raw_train,    X_raw_temp,
    y_train,        y_temp
) = train_test_split(
    X_tfidf, X_bilstm, X_raw, y,
    test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

# Second split: 15% val, 15% test (50% of temp)
(
    X_tfidf_val,  X_tfidf_test,
    X_bilstm_val, X_bilstm_test,
    X_raw_val,    X_raw_test,
    y_val,        y_test
) = train_test_split(
    X_tfidf_temp, X_bilstm_temp, X_raw_temp, y_temp,
    test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print(f'Train : {len(y_train):,}  ({len(y_train)/len(y)*100:.1f}%)')
print(f'Val   : {len(y_val):,}   ({len(y_val)/len(y)*100:.1f}%)')
print(f'Test  : {len(y_test):,}   ({len(y_test)/len(y)*100:.1f}%)')
print(f'\nTrain class dist: {dict(y_train.value_counts())}')
print(f'Test  class dist: {dict(y_test.value_counts())}')

Train : 20,031  (70.0%)
Val   : 4,293   (15.0%)
Test  : 4,293   (15.0%)

Train class dist: {0: 10488, 1: 9543}
Test  class dist: {0: 2248, 1: 2045}


## 6. TF-IDF Vectorization (for Traditional ML)

In [9]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 3),    # unigrams, bigrams, trigrams
    sublinear_tf=True,
    min_df=2,
    analyzer='word'
)

X_tfidf_train_vec = tfidf.fit_transform(X_tfidf_train)
X_tfidf_val_vec   = tfidf.transform(X_tfidf_val)
X_tfidf_test_vec  = tfidf.transform(X_tfidf_test)

print(f'TF-IDF shape — Train: {X_tfidf_train_vec.shape}')
print(f'TF-IDF shape — Test : {X_tfidf_test_vec.shape}')
print(f'Vocabulary size     : {len(tfidf.vocabulary_):,}')

TF-IDF shape — Train: (20031, 19923)
TF-IDF shape — Test : (4293, 19923)
Vocabulary size     : 19,923


## 7. Sequence Tokenization (for BiLSTM)

Since headlines are only 8–15 words, **MAX_LEN = 40** covers 99%+ of all headlines with zero truncation issues — this was the root cause of failure in our previous project.

In [16]:
MAX_WORDS = 30000   # vocabulary size — larger because dataset is bigger
MAX_LEN   = 40      # 99th percentile of headline word count

keras_tok = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
keras_tok.fit_on_texts(X_bilstm_train)

X_seq_train = pad_sequences(
    keras_tok.texts_to_sequences(X_bilstm_train),
    maxlen=MAX_LEN, padding='post', truncating='post'
)
X_seq_val = pad_sequences(
    keras_tok.texts_to_sequences(X_bilstm_val),
    maxlen=MAX_LEN, padding='post', truncating='post'
)
X_seq_test = pad_sequences(
    keras_tok.texts_to_sequences(X_bilstm_test),
    maxlen=MAX_LEN, padding='post', truncating='post'
)

print(f'Sequence shape — Train: {X_seq_train.shape}')
print(f'Sequence shape — Test : {X_seq_test.shape}')
print(f'MAX_WORDS: {MAX_WORDS}, MAX_LEN: {MAX_LEN}')

# Verify: check how many sequences are truncated
zero_ratio = (X_seq_train == 0).mean(axis=1)
print(f'\nAvg padding per sequence : {zero_ratio.mean()*100:.1f}%')
print(f'Sequences truncated      : {(zero_ratio == 0).sum()} (should be near 0)')

Sequence shape — Train: (20031, 40)
Sequence shape — Test : (4293, 40)
MAX_WORDS: 30000, MAX_LEN: 40

Avg padding per sequence : 74.9%
Sequences truncated      : 0 (should be near 0)


## 8. Save Everything

In [17]:
from joblib import dump

# Convert sparse matrices to dense to avoid pickle issues
splits = {
    'X_tfidf_train': X_tfidf_train_vec.toarray(),  # Convert to dense
    'X_tfidf_val'  : X_tfidf_val_vec.toarray(),
    'X_tfidf_test' : X_tfidf_test_vec.toarray(),
    'X_seq_train'  : X_seq_train,
    'X_seq_val'    : X_seq_val,
    'X_seq_test'   : X_seq_test,
    'X_raw_train'  : list(X_raw_train),
    'X_raw_val'    : list(X_raw_val),
    'X_raw_test'   : list(X_raw_test),
    'y_train'      : y_train.values,
    'y_val'        : y_val.values,
    'y_test'       : y_test.values,
}

params = {'MAX_WORDS': MAX_WORDS, 'MAX_LEN': MAX_LEN}

dump(splits,             '../data/splits.joblib')
dump(tfidf,              '../data/tfidf.joblib')
dump(keras_tok,          '../data/keras_tok.joblib')
dump(params,             '../data/lstm_params.joblib')

print('All files saved!')
print(f'  splits.joblib    — all train/val/test splits (dense arrays)')
print(f'  tfidf.joblib     — fitted TF-IDF vectorizer')
print(f'  keras_tok.joblib — fitted Keras tokenizer')
print(f'  lstm_params.joblib — MAX_WORDS & MAX_LEN')

All files saved!
  splits.joblib    — all train/val/test splits (dense arrays)
  tfidf.joblib     — fitted TF-IDF vectorizer
  keras_tok.joblib — fitted Keras tokenizer
  lstm_params.joblib — MAX_WORDS & MAX_LEN


---
## ✅ Preprocessing Summary

| Step | Detail |
|------|--------|
| Lowercasing | Applied to all text |
| URL / noise removal | Applied |
| Stopwords | Removed (kept negation words) |
| Lemmatization | Applied for TF-IDF |
| TF-IDF | 20,000 features, unigrams–trigrams |
| MAX_LEN | 40 — covers 99%+ of headlines with NO truncation |
| Split | 70% train / 15% val / 15% test, stratified |

**Next:** Traditional ML models → `03_traditional_ml.ipynb`